In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
%%capture
!pip install camel-tools
!pip install -q urlextract
!pip install PyArabic

In [3]:
import pandas as pd
import re

from camel_tools.utils.normalize import normalize_unicode
from camel_tools.utils.normalize import normalize_alef_maksura_ar
from camel_tools.utils.normalize import normalize_alef_ar
from camel_tools.utils.normalize import normalize_teh_marbuta_ar
from camel_tools.utils.dediac import dediac_ar

from urlextract import URLExtract

from pyarabic.araby import strip_tatweel

In [4]:
%%capture
!git clone https://github.com/NoraAlt/Mawqif-Arabic-Stance.git

In [5]:
arabic_to_latin_numbers = { '٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4', '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9' }

In [6]:
arabic_to_latin_numbers = { '٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4', '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9' }

extractor = URLExtract()

def clean_text(text):
  text = normalize_unicode(text) # ﷺ -> صلى الله عليه وسلم
  text = normalize_alef_ar(text) # Normalize alef variants to 'ا'
  text = normalize_alef_maksura_ar(text) # Normalize alef maksura 'ى' to yeh 'ي'
  text = normalize_teh_marbuta_ar(text) # Normalize teh marbuta 'ة' to heh 'ه'
  text = dediac_ar(text) # Dediacritization
  text = text.translate(str.maketrans(arabic_to_latin_numbers))

  urls = extractor.find_urls(text)
  for url in urls:
      text = text.replace(url, '')

  text = strip_tatweel(text)    # e.g. الســـــــــــــــــــعودية -> السعودية

  text = re.sub(r'(.)\1{2,}', r'\1\1', text)  # Replace 3+ consecutive chars with just 2

  # text = re.sub(r'[^ء-يa-zA-Z0-9\s\(\)\[\]\{\}]+', ' ', text) # Remove all characters except Arabic letters, English letters, digits, # whitespace, and specific punctuation marks (parentheses, brackets, braces).
  text = re.sub(r'[^ء-ي\s]+', ' ', text)
  text = re.sub(r'\s+', ' ', text).strip() # Collapse multiple spaces into a single space and trim leading/trailing spaces.

  return text


In [7]:
def print_df_info(name, df):
  print('*'*5)
  acopy = df.copy()
  print(f'len({name}) = {len(df)}')
  print('')
  print(acopy.groupby(['stance']).size().reset_index(name='count'))
  print('')
  print(acopy.groupby(['sarcasm']).size().reset_index(name='count'))
  print('')
  print(acopy.groupby(['sentiment']).size().reset_index(name='count'))
  print('')
  print(acopy.groupby('combined_label').size().reset_index(name='count').sort_values(by='count'))
  print('')
  print('*'*5)

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split

training = pd.read_csv('/content/Mawqif-Arabic-Stance/Data/Mawqif_AllTargets_Train.csv')
testing = pd.read_csv('/content/Mawqif-Arabic-Stance/Data/Mawqif_AllTargets_Test.csv')

training.replace({'sarcasm': {'No': False, 'Yes': True}, 'sentiment': {'Negative': 'NEG', 'Positive': 'POS', 'Neutral': 'NEU'}}, inplace=True)
testing.replace({'sarcasm': {'No': False, 'Yes': True}, 'sentiment': {'Negative': 'NEG', 'Positive': 'POS', 'Neutral': 'NEU'}}, inplace=True)

training = training.dropna(subset=['stance']).dropna(subset=['sarcasm']).dropna(subset=['sentiment'])
training = training[training['target'] == 'Covid Vaccine']
training = training[['text', 'stance', 'sarcasm', 'sentiment']].reset_index(drop=True)

testing = testing.dropna(subset=['stance']).dropna(subset=['sarcasm']).dropna(subset=['sentiment'])
testing = testing[testing['target'] == 'Covid Vaccine']
testing = testing[['text', 'stance', 'sarcasm', 'sentiment']].reset_index(drop=True)

training['combined_label'] = training['stance'].astype(str) + ',' + training['sarcasm'].astype(str) + ',' + training['sentiment'].astype(str)
testing['combined_label'] = testing['stance'].astype(str) + ',' + testing['sarcasm'].astype(str) + ',' + testing['sentiment'].astype(str)

training['text'] = training['text'].apply(clean_text)
testing['text'] = testing['text'].apply(clean_text)

training = training[training['text'].str.strip() != ''].dropna(subset=['text'])
testing = testing[testing['text'].str.strip() != ''].dropna(subset=['text'])

training, validation = train_test_split(
    training,
    test_size=0.30,
    stratify=training['stance'],
    random_state=42
)

training = training.reset_index(drop=True)
validation = validation.reset_index(drop=True)
testing = testing.reset_index(drop=True)

print_df_info('training', training)
print_df_info('validation', validation)
print_df_info('testing', testing)

/tmp/ipykernel_16674/4247344483.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  training.replace({'sarcasm': {'No': False, 'Yes': True}, 'sentiment': {'Negative': 'NEG', 'Positive': 'POS', 'Neutral': 'NEU'}}, inplace=True)
/tmp/ipykernel_16674/4247344483.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  testing.replace({'sarcasm': {'No': False, 'Yes': True}, 'sentiment': {'Negative': 'NEG', 'Positive': 'POS', 'Neutral': 'NEU'}}, inplace=True)


*****
len(training) = 711

    stance  count
0  Against    355
1    Favor    356

   sarcasm  count
0    False    662
1     True     49

  sentiment  count
0       NEG    200
1       NEU    307
2       POS    204

      combined_label  count
9     Favor,True,POS      1
2  Against,False,POS      4
5    Favor,False,NEG     11
4   Against,True,NEU     11
8     Favor,True,NEU     15
3   Against,True,NEG     22
6    Favor,False,NEU    130
1  Against,False,NEU    151
0  Against,False,NEG    167
7    Favor,False,POS    199

*****
*****
len(validation) = 305

    stance  count
0  Against    152
1    Favor    153

   sarcasm  count
0    False    284
1     True     21

  sentiment  count
0       NEG     83
1       NEU    142
2       POS     80

      combined_label  count
2  Against,False,POS      2
5    Favor,False,NEG      3
4   Against,True,NEU      4
8     Favor,True,NEU      5
9     Favor,True,POS      5
3   Against,True,NEG      7
1  Against,False,NEU     66
6    Favor,False,NEU     67
7  

In [11]:
training.to_csv('/content/drive/MyDrive/Project/Mawqif/01_data_preparation/train.csv', index=False)
validation.to_csv('/content/drive/MyDrive/Project/Mawqif/01_data_preparation/val.csv', index=False)
testing.to_csv('/content/drive/MyDrive/Project/Mawqif/01_data_preparation/test.csv', index=False)